# 24 · Packaging & Project Structure with `uv`

Notebooks are for learning; **real pipelines live in `.py` modules** inside a
proper project you can import, test, version and run. This notebook shows how a
Python project is laid out, how `uv` manages it, and builds a tiny importable
package live so the ideas are concrete.

## The standard layout

A typical data project looks like this:

```
my-pipeline/
├── pyproject.toml        # project metadata + dependencies (uv reads this)
├── uv.lock               # exact pinned versions (commit this)
├── README.md
├── .python-version       # which Python uv should use
├── src/
│   └── my_pipeline/
│       ├── __init__.py   # marks the folder as an importable package
│       ├── extract.py
│       ├── transform.py
│       └── load.py
└── tests/
    └── test_transform.py
```

The `src/` layout keeps importable code separate from config and tests. A
folder becomes a **package** when it contains `__init__.py`.

## What `pyproject.toml` declares

`pyproject.toml` is the modern standard (it replaces `requirements.txt` +
`setup.py`). It names the project, the required Python, and dependencies:

```toml
[project]
name = "my-pipeline"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
    "pandas>=2.1.0",
    "requests>=2.31.0",
]

[project.scripts]
run-pipeline = "my_pipeline.load:main"   # creates a CLI command
```

## The `uv` workflow

`uv` is a fast, all-in-one project and environment manager. The commands you'll
actually use:

| Command | What it does |
|---|---|
| `uv init my-pipeline` | scaffold a new project |
| `uv add pandas requests` | add dependencies (updates `pyproject.toml` + lock) |
| `uv sync` | create/refresh the `.venv` from the lock file |
| `uv run python -m my_pipeline` | run code inside the managed environment |
| `uv run pytest` | run your tests in that environment |

`uv` creates an isolated `.venv` per project, so dependencies never collide
between projects — and `uv.lock` makes installs reproducible for everyone.

## Build a package live

Let's create a minimal package on disk, then **import and use it** — proving how
modules and packages actually work.

In [ ]:
import sys, tempfile, textwrap
from pathlib import Path

root = Path(tempfile.mkdtemp()) / 'demo_pkg_project' / 'src'
pkg = root / 'demo_pipeline'
pkg.mkdir(parents=True)

(pkg / '__init__.py').write_text('')      # makes it a package
(pkg / 'transform.py').write_text(textwrap.dedent('''
    def clean_country(value):
        """Normalize a country code."""
        return (value or "").strip().upper() or "UNKNOWN"
'''))
print('created:', *(p.name for p in pkg.iterdir()))

In [ ]:
# Put src/ on the import path (uv/editable installs do this for you),
# then import the package like any real dependency.
sys.path.insert(0, str(root))
from demo_pipeline.transform import clean_country

print(clean_country('  us '))
print(clean_country(''))

## Running as a module: `__main__`

Give a package a `__main__.py` (or a function referenced in
`[project.scripts]`) and it runs with `python -m package` or your CLI command —
the entry point a scheduler calls. Combine with `argparse` (the logging & CLI notebook) for
parameters.

In [ ]:
import textwrap
(pkg / '__main__.py').write_text(textwrap.dedent('''
    from demo_pipeline.transform import clean_country
    def main():
        print("pipeline running:", clean_country(" gb "))
    if __name__ == "__main__":
        main()
'''))

import subprocess, sys
out = subprocess.run([sys.executable, '-m', 'demo_pipeline'],
                     cwd=str(root), capture_output=True, text=True)
print(out.stdout.strip() or out.stderr.strip())

### Recap

Real code lives in an importable **package** (`src/my_pkg/` with `__init__.py`),
configured by **`pyproject.toml`** and managed by **`uv`** (`uv add`, `uv sync`,
`uv run`); `uv.lock` pins versions for reproducibility; `python -m package` (or
a `[project.scripts]` entry point) is how a pipeline is launched. That completes
the **Python bootcamp** — you can write, structure, test and ship real Python.
For dataframe wrangling and a full ETL capstone, continue to the companion
**pandas-numpy-bootcamp**.